# 03 - Random Forest Regressor

Random Forest is useful for tabular datasets because it can capture nonlinear relationships and feature interactions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [ ]:
DATA_DIR = Path('../data')
MODEL_DIR = Path('../outputs/models')
RESULT_DIR = Path('../outputs/results')

math_df = pd.read_csv(DATA_DIR / 'student-mat.csv', sep=';')
por_df = pd.read_csv(DATA_DIR / 'student-por.csv', sep=';')
math_df['subject'] = 'Math'
por_df['subject'] = 'Portuguese'
df = pd.concat([math_df, por_df], ignore_index=True).drop_duplicates()

X = df.drop(columns=['G3'])
y = df['G3']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

In [ ]:
preprocessor = ColumnTransformer([
    ('num', 'passthrough', numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

param_grid = {
    'model__n_estimators': [80, 120],
    'model__max_depth': [None, 10],
    'model__min_samples_leaf': [1, 2]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=1
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
predictions = best_model.predict(X_test)

print('Best parameters:', grid_search.best_params_)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

metrics = pd.DataFrame([{
    'Model': 'Random Forest Regressor',
    'RMSE': rmse,
    'MAE': mae,
    'R2': r2,
    'Best Parameters': str(grid_search.best_params_)
}])

display(metrics)
metrics.to_csv(RESULT_DIR / 'random_forest_metrics.csv', index=False)
joblib.dump(best_model, MODEL_DIR / 'random_forest_model.pkl')

In [ ]:
feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()
importances = best_model.named_steps['model'].feature_importances_

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False).head(20)

display(importance_df)

plt.figure(figsize=(9, 6))
plt.barh(importance_df['feature'][::-1], importance_df['importance'][::-1])
plt.title('Random Forest Top Feature Importance')
plt.show()